# 03 — Per-Wallet Rolling Sortino

Goal: from hourly NAV series per wallet, compute rolling 90d Sortino + drawdown gating signals.

Output: `data/wallet_sortino_hourly.parquet` with schema `(timestamp_hour, wallet, rolling_90d_sortino, realized_dd_30d, n_observed_positions_90d)`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import pandas as pd
import numpy as np
import plotly.express as px
from sortino import rolling_sortino, max_drawdown

In [ ]:
# Load reconstructed NAV from notebook 02
# nav = pd.read_parquet('../data/wallet_nav_hourly.parquet')
# nav.head()

## 1. Hourly returns per wallet

In [ ]:
def compute_hourly_returns(nav_df: pd.DataFrame) -> pd.DataFrame:
    nav_df = nav_df.sort_values(['wallet', 'timestamp_hour'])
    nav_df['return_hourly'] = (
        nav_df.groupby('wallet')['position_value_usd'].pct_change()
    )
    return nav_df

## 2. Rolling 90d Sortino per wallet

Window: 90 days × 24 hours = 2160 periods. Min observations: 20.

In [ ]:
WINDOW_PERIODS = 90 * 24
MIN_OBS = 20

def compute_rolling_sortino(returns_df: pd.DataFrame) -> pd.DataFrame:
    out = []
    for wallet, group in returns_df.groupby('wallet'):
        series = group.set_index('timestamp_hour')['return_hourly'].dropna()
        rolling = rolling_sortino(series, window_periods=WINDOW_PERIODS, min_observations=MIN_OBS)
        df = rolling.rename('rolling_90d_sortino').reset_index()
        df['wallet'] = wallet
        out.append(df)
    return pd.concat(out, ignore_index=True)

## 3. Realized 30d drawdown per wallet

In [ ]:
def compute_rolling_dd(nav_df: pd.DataFrame, window_days: int = 30) -> pd.DataFrame:
    window_periods = window_days * 24
    out = []
    for wallet, group in nav_df.groupby('wallet'):
        series = group.set_index('timestamp_hour')['position_value_usd']
        rolling_max = series.rolling(window=window_periods, min_periods=24).max()
        dd = (series - rolling_max) / rolling_max
        df = (-dd).rename('realized_dd_30d').reset_index()
        df['wallet'] = wallet
        out.append(df)
    return pd.concat(out, ignore_index=True)

## 4. Sample size guard — count observed positions in 90d window

In [ ]:
def compute_observation_count(positions_df: pd.DataFrame) -> pd.DataFrame:
    """For each (wallet, timestamp_hour), count distinct non-zero position rows
    observed in the trailing 90 days. Feeds the >=20 observation gate.
    """
    window_periods = 90 * 24
    positions_df = positions_df.assign(
        has_position=(positions_df['cumulative_amount'] > 0).astype(int)
    )
    out = []
    for wallet, group in positions_df.groupby('wallet'):
        series = group.set_index('timestamp_hour')['has_position']
        counts = series.rolling(window=window_periods, min_periods=1).sum()
        df = counts.rename('n_observed_positions_90d').reset_index()
        df['wallet'] = wallet
        out.append(df)
    return pd.concat(out, ignore_index=True)

## 5. Merge + persist

Final schema: (timestamp_hour, wallet, rolling_90d_sortino, realized_dd_30d, n_observed_positions_90d) — direct input to the classifier.